# Assignment Text summariser powered by LLM

In [39]:
# Importing packages
import numpy as np
import torch
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq
import evaluate

## Checking prerequisites

In [40]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))
print("PyTorch version:", torch.__version__)

CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3050 Laptop GPU
PyTorch version: 2.2.2+cu121


In [41]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [42]:
import torch
print("Torch version:", torch.__version__)
print("Torch location:", torch.__file__)

import transformers
print("Transformers version:", transformers.__version__)

Torch version: 2.2.2+cu121
Torch location: c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\torch\__init__.py
Transformers version: 4.40.0


### Loading Dataset

In [44]:
dataset = load_from_disk("S:\Projects\Datasets\TextS\samsum_dataset")

#### Neural networks cannot process raw text directly.
#### Text must first be converted into numerical representations.
#### Modern LLMs achieve this using subword tokenization.

### LLM's like gpt rely on tokenization methods like BPE, but for this dataset and T5 llm we will use SentencePiece

### Lets take a look at token embedding and trannsformer, Text to text transfer transformer, Flan-T5-base (trained on 250M parameters), big enough to be called LLM

In [52]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### Lets inspect the model

In [53]:
model.config

T5Config {
  "_name_or_path": "google/flan-t5-base",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": true,
      "

In [54]:
model.shared

Embedding(32128, 768)

In [55]:
model.encoder

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerFF(
          (DenseReluDense): T5DenseGatedActDense(
            (wi_0): Linear(in_features=768, out_features=2048, bias=False)
            (wi_1): Linear(in_features=768, out_features=2048, bias=False)
            (wo): Linear(in_features=2048, out_features=768, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
      

In [56]:
model.decoder

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerCrossAttention(
          (EncDecAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=F

In [57]:
model.lm_head

Linear(in_features=768, out_features=32128, bias=False)

### Also the tokenizer

In [58]:
tokenizer.vocab_size

32100

In [59]:
tokenizer.model_max_length

512

In [60]:
tokenizer.pad_token

'<pad>'

In [61]:
tokenizer.eos_token

'</s>'

In [62]:
sample = dataset["train"][0]["dialogue"]
print(sample)

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)


In [63]:
tokens = tokenizer.tokenize(sample)
print(tokens[:50])

['▁Amanda', ':', '▁I', '▁baked', '▁cookies', '.', '▁Do', '▁you', '▁want', '▁some', '?', '▁Jerry', ':', '▁Sure', '!', '▁Amanda', ':', '▁I', "'", 'll', '▁bring', '▁you', '▁tomorrow', '▁', ':', '-', ')']


In [64]:
ids = tokenizer.encode(sample)
print(ids[:50])

[21542, 10, 27, 13635, 5081, 5, 531, 25, 241, 128, 58, 16637, 10, 10625, 55, 21542, 10, 27, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1]


### model already has pretrtained embedding lets extract one

In [69]:
embeddings = model.get_input_embeddings().weight
print(embeddings.shape)

torch.Size([32128, 768])


In [73]:
tokens = tokenizer.tokenize("hello")
print(tokens)

token = tokens[0]

token_id = tokenizer.convert_tokens_to_ids(token)

print(token)
print(token_id)

['▁hello']
▁hello
21820


#### The pretrainied embedding has 21820 as hello, i will make more sense when compared with surounding vectors and it will have closer semantic meaning

### T5 base dont need a positional embedding, its has Relative position, biased inside the attention mechanism

## Now lets take a look at attenetion block

In [75]:
print(model.encoder.block[0].layer[0])

T5LayerSelfAttention(
  (SelfAttention): T5Attention(
    (q): Linear(in_features=768, out_features=768, bias=False)
    (k): Linear(in_features=768, out_features=768, bias=False)
    (v): Linear(in_features=768, out_features=768, bias=False)
    (o): Linear(in_features=768, out_features=768, bias=False)
    (relative_attention_bias): Embedding(32, 12)
  )
  (layer_norm): T5LayerNorm()
  (dropout): Dropout(p=0.1, inplace=False)
)


In [78]:
attn = model.encoder.block[0].layer[0].SelfAttention
print(attn)

T5Attention(
  (q): Linear(in_features=768, out_features=768, bias=False)
  (k): Linear(in_features=768, out_features=768, bias=False)
  (v): Linear(in_features=768, out_features=768, bias=False)
  (o): Linear(in_features=768, out_features=768, bias=False)
  (relative_attention_bias): Embedding(32, 12)
)


In [79]:
print(attn.has_relative_attention_bias)

True


##### This shows that the model has relative attention type, rather than absolute that we see in classic gpt

In [80]:
print(attn.relative_attention_bias)

Embedding(32, 12)


#### Here 32 is the relative distance buckets with 12 attention heads, model learns distance rather than positions, which is a smarter approach

In [ ]:
# Attention weights
attn.relative_attention_bias.weight

Parameter containing:
tensor([[ 3.3072e+00, -1.4124e+01,  2.2363e+00, -7.5515e+00,  8.4037e+00,
          5.4025e+00,  4.9113e-01,  2.5243e-01,  4.3401e+00,  6.6022e+00,
         -8.6801e+00, -2.5473e+01],
        [-2.5756e+01,  1.0481e+01,  8.4726e+00,  3.9471e+00,  9.8540e+00,
          1.7485e+00,  9.1644e+00,  6.1179e+00,  7.9472e+00, -4.2284e+00,
          2.8060e+00,  7.6758e+00],
        [-1.5956e+01,  8.7715e+00,  5.2965e+00,  4.5750e+00,  7.7746e+00,
          9.5001e-01,  8.6429e+00,  6.6384e+00,  7.5241e+00, -1.7510e+01,
          3.7001e+00,  8.0501e+00],
        [-1.5508e+01,  7.6623e+00,  4.6198e+00,  4.7793e+00,  6.7673e+00,
          1.9559e+00,  8.1014e+00,  6.7059e+00,  7.0760e+00, -1.9015e+01,
          3.9715e+00,  8.0325e+00],
        [-1.3945e+01,  7.0022e+00,  4.4271e+00,  4.7923e+00,  5.8716e+00,
          2.2086e+00,  7.6472e+00,  6.8344e+00,  6.7654e+00, -2.1642e+01,
          4.1278e+00,  7.9277e+00],
        [-1.5966e+01,  6.4181e+00,  4.3777e+00,  4.9517e+0

#### These are pretrained weights